In [ ]:
#Loaded the Files:

#FINAL_NSCLC_EXPRESSION.csv
#CCLE_mutations.csv
#GDSC_FULL_SMILES.csv
#TASK_AWARE_CELL_FEATURES.csv
#BEST_END_TO_END_DRUG_GNN_230_EPOCHS.pth

In [ ]:
import os

for f in os.listdir():
    print(f)

.config
TASK_AWARE_CELL_FEATURES.csv
FINAL_NSCLC_EXPRESSION.csv
GDSC_FULL_SMILES.csv
BEST_END_TO_END_DRUG_GNN_230_EPOCHS.pth
CCLE_mutations.csv
sample_data


In [ ]:
!pip install torch-geometric rdkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 48.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from rdkit import Chem

from torch_geometric.data import Data
from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)

In [ ]:
cell_df = pd.read_csv(
    "TASK_AWARE_CELL_FEATURES.csv"
)

smiles_df = pd.read_csv(
    "GDSC_FULL_SMILES.csv"
)

print(cell_df.shape)
print(smiles_df.shape)

cell_df.head()

(92, 273)
(229, 2)


,ModelID,0,1,2,3,4,5,6,7,8,...,PIK3CA,ALK,ROS1,MET,ERBB2,NF1,RB1,SMARCA4,CDKN2A,PTEN
0,ACH-000769,0.130234,0.000000,0.000000,0.976543,0.0,0.000000,1.669606,0.291935,0.000000,...,0,0,0,0,0,0,0,0,1,0
1,ACH-000528,0.603273,1.044293,0.000000,0.000000,0.0,0.240411,1.687555,0.044271,0.000000,...,0,1,0,0,0,1,0,0,0,0
2,ACH-000585,0.050205,0.000000,0.000000,0.226873,0.0,0.496220,0.726164,0.985220,0.455805,...,0,0,0,0,0,1,0,1,0,0
3,ACH-000176,0.000000,0.000000,0.122873,0.000000,0.0,0.000000,0.567646,0.584774,0.232363,...,1,0,0,0,0,0,0,0,0,0
4,ACH-000587,0.360172,0.000000,0.000000,0.000000,0.0,0.000000,1.394738,1.188975,0.000000,...,1,0,0,0,0,0,0,0,1,0


In [ ]:
from rdkit import Chem
from torch_geometric.data import Data

def mol_to_graph(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    node_features = []

    for atom in mol.GetAtoms():

        node_features.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetIsAromatic()),
            atom.GetTotalNumHs()
        ])

    edge_index = []

    for bond in mol.GetBonds():

        start = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()

        edge_index.append([start, end])
        edge_index.append([end, start])

    x = torch.tensor(
        node_features,
        dtype=torch.float
    )

    edge_index = torch.tensor(
        edge_index,
        dtype=torch.long
    ).t().contiguous()

    return Data(
        x=x,
        edge_index=edge_index
    )

drug_graphs = {}

for _, row in smiles_df.iterrows():

    graph = mol_to_graph(
        row["CanonicalSMILES"]
    )

    if graph is not None:

        drug_graphs[
            row["Drug"]
        ] = graph

print("Graphs:", len(drug_graphs))

Graphs: 229


In [ ]:
class DrugGNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(5, 64)
        self.conv2 = GCNConv(64, 128)

        self.fc = nn.Linear(
            128,
            256
        )

    def forward(
        self,
        x,
        edge_index,
        batch
    ):

        x = torch.relu(
            self.conv1(
                x,
                edge_index
            )
        )

        x = torch.relu(
            self.conv2(
                x,
                edge_index
            )
        )

        x = global_mean_pool(
            x,
            batch
        )

        x = self.fc(x)

        return x


class DrugResponseGNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.drug_gnn = DrugGNN()

        self.cell_fc = nn.Linear(
            272,
            256
        )

        self.fc1 = nn.Linear(
            512,
            512
        )

        self.fc2 = nn.Linear(
            512,
            256
        )

        self.fc3 = nn.Linear(
            256,
            128
        )

        self.out = nn.Linear(
            128,
            1
        )

        self.dropout = nn.Dropout(
            0.3
        )

    def forward(
        self,
        cell_features,
        graph_batch
    ):

        drug_embedding = self.drug_gnn(
            graph_batch.x.float(),
            graph_batch.edge_index,
            graph_batch.batch
        )

        cell_embedding = torch.relu(
            self.cell_fc(
                cell_features
            )
        )

        x = torch.cat(
            [
                cell_embedding,
                drug_embedding
            ],
            dim=1
        )

        x = torch.relu(
            self.fc1(x)
        )

        x = self.dropout(x)

        x = torch.relu(
            self.fc2(x)
        )

        x = self.dropout(x)

        x = torch.relu(
            self.fc3(x)
        )

        x = self.out(x)

        return x.squeeze()

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = DrugResponseGNN().to(device)

model.load_state_dict(
    torch.load(
        "BEST_END_TO_END_DRUG_GNN_230_EPOCHS.pth",
        map_location=device
    )
)

model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


In [ ]:
cell_df["ModelID"].head()

,ModelID
0,ACH-000769
1,ACH-000528
2,ACH-000585
3,ACH-000176
4,ACH-000587


In [ ]:
selected_cell = "ACH-000769"

In [ ]:
cell_row = cell_df[
    cell_df["ModelID"] == selected_cell
]

cell_features = torch.tensor(
    cell_row.drop(
        columns=["ModelID"]
    ).values,
    dtype=torch.float32
).to(device)

print(cell_features.shape)

torch.Size([1, 272])


In [ ]:
cell_df["ModelID"].head(10)

,ModelID
0,ACH-000769
1,ACH-000528
2,ACH-000585
3,ACH-000176
4,ACH-000587
5,ACH-000627
6,ACH-000392
7,ACH-000035
8,ACH-000894
9,ACH-000638


In [ ]:
from torch_geometric.data import Batch

results = []

model.eval()

with torch.no_grad():

    for drug_name, graph in drug_graphs.items():

        graph_batch = Batch.from_data_list(
            [graph]
        ).to(device)

        pred_ic50 = model(
            cell_features,
            graph_batch
        ).item()

        results.append({
            "Drug": drug_name,
            "Predicted_LN_IC50": pred_ic50
        })

results_df = pd.DataFrame(results)

print("Total Drugs Evaluated:", len(results_df))

Total Drugs Evaluated: 229


In [ ]:
results_df = results_df.sort_values(
    "Predicted_LN_IC50",
    ascending=True
)

results_df.head(10)

,Drug,Predicted_LN_IC50
4,Docetaxel,-4.951285
153,Romidepsin,-4.791728
55,Paclitaxel,-4.733583
204,Vinorelbine,-4.669029
1,Vinblastine,-4.647015
148,Dactinomycin,-3.933460
76,Bortezomib,-3.502833
192,Sepantronium bromide,-3.305562
84,Daporinad,-3.021786
28,Staurosporine,-3.015328


In [ ]:
top10 = results_df.head(10)

print(
    f"Top 10 Recommended Drugs for {selected_cell}"
)

top10.reset_index(
    drop=True,
    inplace=True
)

top10.index = top10.index + 1

top10

Top 10 Recommended Drugs for ACH-000769


,Drug,Predicted_LN_IC50
1,Docetaxel,-4.951285
2,Romidepsin,-4.791728
3,Paclitaxel,-4.733583
4,Vinorelbine,-4.669029
5,Vinblastine,-4.647015
6,Dactinomycin,-3.933460
7,Bortezomib,-3.502833
8,Sepantronium bromide,-3.305562
9,Daporinad,-3.021786
10,Staurosporine,-3.015328


In [ ]:
#Another Test Sample:

In [ ]:
selected_cells = [
    "ACH-000769",
    "ACH-000528",
    "ACH-000585",
    "ACH-000176",
    "ACH-000587"
]

all_results = {}

for cell_id in selected_cells:

    print(f"\nProcessing {cell_id}...")

    recommendations = predict_drugs_for_cell(
        cell_id
    )

    top10 = recommendations.head(10)

    top10.to_csv(
        f"{cell_id}_Top10_Recommendations.csv",
        index=False
    )

    all_results[cell_id] = top10

    print(top10.head(5))

print("\nAll recommendation files saved.")


Processing ACH-000769...
          Drug  Predicted_LN_IC50
1    Docetaxel          -4.951285
2   Romidepsin          -4.791728
3   Paclitaxel          -4.733583
4  Vinorelbine          -4.669029
5  Vinblastine          -4.647015

Processing ACH-000528...
                   Drug  Predicted_LN_IC50
1            Romidepsin          -6.127965
2  Sepantronium bromide          -5.315709
3            Paclitaxel          -4.285753
4           Vinblastine          -3.886961
5          Dactinomycin          -3.829325

Processing ACH-000585...
                   Drug  Predicted_LN_IC50
1            Romidepsin          -5.067570
2          Dactinomycin          -4.565191
3             Docetaxel          -4.473259
4  Sepantronium bromide          -3.970616
5            Bortezomib          -3.744292

Processing ACH-000176...
                   Drug  Predicted_LN_IC50
1            Romidepsin          -5.613036
2            Paclitaxel          -5.070312
3             Docetaxel          -4.981649
4  S

In [ ]:
summary_rows = []

for cell_id in selected_cells:

    recommendations = predict_drugs_for_cell(
        cell_id
    )

    top5 = recommendations.head(5)

    for rank, (_, row) in enumerate(
        top5.iterrows(),
        start=1
    ):

        summary_rows.append({
            "Cell_Line": cell_id,
            "Rank": rank,
            "Drug": row["Drug"],
            "Predicted_LN_IC50": row["Predicted_LN_IC50"]
        })

summary_df = pd.DataFrame(
    summary_rows
)

summary_df.to_csv(
    "TOP5_DRUG_RECOMMENDATION_SUMMARY.csv",
    index=False
)

summary_df.head(20)

,Cell_Line,Rank,Drug,Predicted_LN_IC50
0,ACH-000769,1,Docetaxel,-4.951285
1,ACH-000769,2,Romidepsin,-4.791728
2,ACH-000769,3,Paclitaxel,-4.733583
3,ACH-000769,4,Vinorelbine,-4.669029
4,ACH-000769,5,Vinblastine,-4.647015
5,ACH-000528,1,Romidepsin,-6.127965
6,ACH-000528,2,Sepantronium bromide,-5.315709
7,ACH-000528,3,Paclitaxel,-4.285753
8,ACH-000528,4,Vinblastine,-3.886961
9,ACH-000528,5,Dactinomycin,-3.829325


In [ ]:
for cell_id in selected_cells:

    recommendations = predict_drugs_for_cell(
        cell_id
    )

    recommendations.to_csv(
        f"{cell_id}_ALL_DRUG_PREDICTIONS.csv",
        index=False
    )

print("All predictions saved")

All predictions saved


In [ ]:
all_predictions = []

for cell_id in cell_df["ModelID"]:

    recommendations = predict_drugs_for_cell(
        cell_id
    )

    recommendations["Cell_Line"] = cell_id

    all_predictions.append(
        recommendations
    )

all_predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)

all_predictions_df.to_csv(
    "ALL_CELL_LINE_DRUG_PREDICTIONS.csv",
    index=False
)

print(all_predictions_df.shape)

(21068, 3)
